# Capstone #2 — Multi-Agent Research Team

*Notebook 18*

Put it all together.

Combine orchestration, parallel execution, built-in tools, and critique into a full multi-agent research pipeline.
<br>
<br>
**Topics:**

- Coordinating specialists with a procedural pipeline

- Parallel researcher and analyst execution

- Partial-failure recovery — pipeline continues when one specialist fails

- Writer + Critic loop for quality improvement

- Full pipeline: question → research → analysis → draft → critique → report

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, CodeInterpreterTool, Runner, WebSearchTool

# Notebook-specific imports
import asyncio
import time

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"


print("✅ Setup complete")

---

## 🏗️ System Architecture

This capstone combines built-in tools from Week 2 with orchestration patterns from Week 3:

- **Procedural orchestration** — Python coordinates the phases directly with `asyncio` and sequential `Runner.run()` calls. We coordinate phases in code here because the pipeline shape is fixed and we want explicit control over parallel execution and partial-failure handling — handoffs and agents-as-tools are better when the agent itself decides what to do next.
- **Lesson 16: Parallel Execution** — researcher and analyst run simultaneously
- **Lesson 17: Debate & Critique** — writer produces a draft, critic reviews, writer revises

**Pipeline:**
```
Research Question
        ↓
Pipeline coordinates specialists
   ↓               ↓
Researcher    Analyst      ← parallel execution
(web search)  (code interp)
   ↓               ↓
      Writer  ← receives both outputs
        ↓
      Critic  ← reviews and flags issues
        ↓
   Final Report
```

**Reliability note:** The pipeline uses `asyncio.gather(return_exceptions=True)` for parallel steps. If one specialist fails, the pipeline detects it, logs the failure, and continues with partial results rather than crashing.

---

## 🤖 Phase 1: Build the Specialist Agents

## 🤖 Build the Specialist Agents

In [ ]:
# Researcher — web search for current information
researcher_instructions = (
    "You are a research specialist. Search the web to find:\n"
    "- Current trends and recent developments\n"
    "- Key statistics and data points\n"
    "- Expert opinions and notable examples\n"
    "Summarize findings clearly with source attribution."
)

researcher_agent = Agent(
    name="Researcher",
    instructions=researcher_instructions,
    model=MODEL,
    tools=[WebSearchTool()]
)

# Analyst — code interpreter for quantitative analysis
analyst_instructions = (
    "You are a data analyst. When given data or statistics:\n"
    "- Use code to compute derived metrics\n"
    "- Identify patterns and trends\n"
    "- Produce clear numerical summaries\n"
    "Always show your calculations."
)

# container: "auto" lets the SDK reuse a Code Interpreter sandbox across calls — faster and cheaper than spinning a fresh one each time
analyst_agent = Agent(
    name="Analyst",
    instructions=analyst_instructions,
    model=MODEL,
    tools=[CodeInterpreterTool(tool_config={
        "type": "code_interpreter",
        "container": {"type": "auto"}
    })]
)

# Writer — produces structured report from research
writer_instructions = (
    "You are a research report writer. Given research findings and analysis:\n"
    "- Write a structured report with clear sections\n"
    "- Use specific data points and cite sources\n"
    "- Keep language clear and professional\n"
    "Format: Executive Summary, Key Findings, Data Analysis, Conclusion"
)

writer_agent = Agent(
    name="Writer",
    instructions=writer_instructions,
    model=MODEL
)

# Critic — reviews and challenges the report
# We give the Critic REASONING_MODEL because critique is judgment-heavy —
# the other specialists do well-scoped work and run on MODEL.
critic_instructions = (
    "You are a critical editor for research reports. Review the report and identify:\n"
    "- Unsupported claims or missing evidence\n"
    "- Logical gaps or weak conclusions\n"
    "- Missing context or important omissions\n"
    "Be direct and specific. Do not rewrite — only critique."
)

critic_agent = Agent(
    name="Critic",
    instructions=critic_instructions,
    model=REASONING_MODEL
)

# --------------------------------------------------------------
print("✅ Specialist agents ready")
print("    Researcher (web search)")
print("    Analyst (code interpreter)")
print("    Writer")
print("    Critic")

## 🚀 Phase 2: Run the Full Pipeline

We'll orchestrate the pipeline directly in code — parallel research, sequential writing and critique. This gives us full visibility into each step.

## 🚀 The Research Pipeline

We'll orchestrate the pipeline directly in code — parallel research, sequential writing and critique. This gives us full visibility into each step.

This function is the reusable pattern: run independent specialists in parallel, convert failures into fallback text, then feed their outputs into downstream agents one phase at a time.

In [ ]:
async def research_pipeline(question):
    """Full multi-agent research pipeline."""

    print(f"🔬 Research Question: {question}")
    print("=" * 60)
    start = time.time()

    # -------------------------------------------------------
    # Phase 1: Parallel research and analysis
    # -------------------------------------------------------
    print("\nPhase 1: Researching and analyzing in parallel...")

    research_result, analysis_result = await asyncio.gather(
        Runner.run(researcher_agent, input=question),
        Runner.run(
            analyst_agent,
            input=f"Analyze the quantitative aspects of this question: {question}"
        ),
        return_exceptions=True
    )

    # Handle partial failures — pipeline continues even if one specialist fails
    if isinstance(research_result, Exception):
        print(f"  ⚠️  Researcher failed: {research_result}")
        research_findings = "Web research unavailable for this run."
    else:
        research_findings = research_result.final_output

    if isinstance(analysis_result, Exception):
        print(f"  ⚠️  Analyst failed: {analysis_result}")
        analysis_findings = "Quantitative analysis unavailable for this run."
    else:
        analysis_findings = analysis_result.final_output

    print(f"  ✓ Research phase complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 2: Write the report
    # -------------------------------------------------------
    print("\nPhase 2: Writing report...")

    writer_input = (
        f"Write a research report on: {question}\n\n"
        f"RESEARCH FINDINGS:\n{research_findings}\n\n"
        f"DATA ANALYSIS:\n{analysis_findings}"
    )

    write_result = await Runner.run(writer_agent, input=writer_input)

    draft = write_result.final_output
    print(f"  ✓ Draft complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 3: Critique the draft
    # -------------------------------------------------------
    print("\nPhase 3: Reviewing draft...")

    critique_result = await Runner.run(critic_agent, input=f"Critique this research report:\n\n{draft}")

    critique = critique_result.final_output
    print(f"  ✓ Critique complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 4: Revise the report
    # -------------------------------------------------------
    print("\nPhase 4: Revising report...")

    revision_input = (
        f"Revise this research report based on the critique.\n\n"
        f"ORIGINAL REPORT:\n{draft}\n\n"
        f"CRITIQUE:\n{critique}\n\n"
        f"Rewrite addressing all feedback."
    )

    revision_result = await Runner.run(writer_agent, input=revision_input)

    final_report = revision_result.final_output

    total_time = time.time() - start
    print(f"  ✓ Revision complete ({total_time:.1f}s total)")

    return {
        "question": question,
        "research": research_findings,
        "analysis": analysis_findings,
        "draft": draft,
        "critique": critique,
        "final_report": final_report,
        "total_time": total_time
    }


# --------------------------------------------------------------
print("✅ research_pipeline() ready")

## 🎬 Phase 3: Demo

⚠️ **Cost note:** This pipeline uses web search, code interpreter, and multiple model calls. Cost and runtime will vary depending on tool behavior and container reuse — budget a few minutes and a small number of API credits per run.

In [ ]:
result = await research_pipeline(
    "What is the current state of electric vehicle adoption globally?"
)
print("\n" + "="*60)
print("FINAL REPORT")
print("="*60)
print(result["final_report"])

Try temporarily breaking one specialist (e.g., pass an invalid tool) to watch the fallback text appear — the pipeline keeps running instead of crashing.

---

## 💪 Practice Exercises

### Exercise 1: Add a Fact Checker

*Covers: pipeline extension — adding a verification phase*

Extend the pipeline with a `FactChecker` agent that verifies the key claims in the final report using web search before it's delivered.

This is the main way you'll customize a pipeline in your own project — keep the orchestration shape, then insert or swap specialist phases where your workflow needs them.

# --------------------------------------------------------------
# 💪 Exercise: Add a Fact Checker to the Pipeline
# --------------------------------------------------------------
# Objective: Add a verification step after the revision is complete.

async def research_pipeline_with_fact_check(question):
    """Research pipeline with fact checking step."""

    print(f"🔬 Research Question: {question}")
    print("=" * 60)
    start = time.time()

    # -------------------------------------------------------
    # Phase 1: Parallel research and analysis (provided)
    # -------------------------------------------------------
    print("\nPhase 1: Researching and analyzing in parallel...")

    results = await asyncio.gather(
        Runner.run(researcher_agent, input=question),
        Runner.run(
            analyst_agent,
            input=f"Analyze the quantitative aspects of this question: {question}"
        ),
        return_exceptions=True
    )

    # If a specialist failed, use fallback text so downstream phases still run
    research_result, analysis_result = results
    if isinstance(research_result, Exception):
        print(f"  ⚠️  Researcher failed: {research_result}")
        research_findings = "Web research unavailable for this run."
    else:
        research_findings = research_result.final_output

    if isinstance(analysis_result, Exception):
        print(f"  ⚠️  Analyst failed: {analysis_result}")
        analysis_findings = "Quantitative analysis unavailable for this run."
    else:
        analysis_findings = analysis_result.final_output
    print(f"  ✓ Research complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 2: Write the report (provided)
    # -------------------------------------------------------
    print("\nPhase 2: Writing report...")

    writer_input = (
        f"Write a research report on: {question}\n\n"
        f"RESEARCH FINDINGS:\n{research_findings}\n\n"
        f"DATA ANALYSIS:\n{analysis_findings}"
    )

    write_result = await Runner.run(writer_agent, input=writer_input)

    draft = write_result.final_output
    print(f"  ✓ Draft complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 3: Critique (provided)
    # -------------------------------------------------------
    print("\nPhase 3: Reviewing draft...")

    critique_result = await Runner.run(critic_agent, input=f"Critique this research report:\n\n{draft}")

    critique = critique_result.final_output
    print(f"  ✓ Critique complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 4: Revise (provided)
    # -------------------------------------------------------
    print("\nPhase 4: Revising report...")

    revision_input = (
        f"Revise this research report based on the critique.\n\n"
        f"ORIGINAL REPORT:\n{draft}\n\n"
        f"CRITIQUE:\n{critique}\n\n"
        f"Rewrite addressing all feedback."
    )

    revision_result = await Runner.run(writer_agent, input=revision_input)

    revised_report = revision_result.final_output
    print(f"  ✓ Revision complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 5 (NEW): Fact check — YOUR CODE HERE
    # -------------------------------------------------------
    print("\nPhase 5: Fact checking...")

    # TODO 1: Create a fact_checker_agent with WebSearchTool
    #          Instructions: identify the 3 most important factual claims
    #          in the report, search to verify each one, and report
    #          which claims are confirmed, unverified, or questionable

    # TODO 2: Run the fact checker with revised_report as input

    # TODO 3: Store the result in fact_check_result

    # TODO 4: Print the fact check findings
    #          and the total elapsed time

    # TODO 5: Return a dict with question, final_report, fact_check,
    #          and total_time
    return {}  # TODO: Replace with your dict


# --- Write your code below this line ---

# --------------------------------------------------------------
# TODO: Test your extended pipeline
# result = await research_pipeline_with_fact_check(
#      "What is the current state of electric vehicle adoption globally?"
# )

print("💡 Implement Phase 5 above, then run the pipeline!")

### Exercise 2: Evaluate the Pipeline Output

*Covers: judge agent pattern — scoring multi-agent pipeline output*

Apply the judge agent pattern from Lesson 09 to score the final report produced by the pipeline.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Evaluate the Pipeline Output
# --------------------------------------------------------------
# Objective: Apply the Lesson 09 judge agent pattern to multi-agent output.
# Note: run the demo cell above first to populate result["final_report"].


# TODO 1: Define a ReportEval Pydantic model with:
#           - score: Annotated[int, Field(ge=1, le=5)]
#           - reasoning: str
#           - passed: bool

# TODO 2: Create a judge agent using REASONING_MODEL with output_type=ReportEval
#           Instructions: score the report on clarity, evidence quality,
#           and logical structure. Score 1-5. passed=True if score >= 3.

# TODO 3: Run the judge agent with result["final_report"] as input

# TODO 4: Print the score, reasoning, and pass/fail verdict

# TODO 5: Save the eval result to a file (e.g. "golden_report_eval.json")
#           so you can rerun this judge against future pipeline outputs
#           and catch regressions after prompt or model changes

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Multi-Agent Pipelines Combine Patterns:**

- Parallel execution cuts Phase 1 time when tasks are independent
- Sequential execution for phases that depend on previous output
- Critique loop improves quality without requiring a single perfect agent

**Use Models Strategically:**

- Stronger models are useful for higher-judgment tasks like evaluation, critique, and scoring
- Well-defined specialist tasks work well with `MODEL` — reserve stronger models for where reasoning depth matters
- Balance quality with cost by matching model capability to task complexity

**The Pipeline is Code:**

- No special SDK orchestration needed — `asyncio.gather()` and sequential `await` calls
- Each phase's output becomes the next phase's input
- Easy to add, remove, or reorder phases

**Parallel Phases Need Failure Handling:**

- `asyncio.gather(return_exceptions=True)` prevents one failure from crashing the pipeline
- Check each result with `isinstance(result, Exception)`
- Provide fallback text so downstream phases always have something to work with

## 📍 Next Step

**Notebook 19: Sessions & Conversation State**  

Give your agents persistent memory so they remember context across separate conversations.

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-18-capstone-2--multi-agent-research-team)

---